In [1]:
import os
from dotenv import load_dotenv
from typing import Any, List, Optional, Dict

load_dotenv()

True

In [2]:
from openai import OpenAI
from pydantic import Field
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, SystemMessage, ChatMessage, HumanMessage


class QwenChatModel(BaseChatModel):
    """
    基于 BaseChatModel 封装的阿里云 Qwen 自定义类。
    支持 enable_thinking 参数以获取思考过程。
    """

    model_name: str = Field(default="qwen-plus", alias="model")
    api_key: Optional[str] = Field(default=None)
    base_url: Optional[str] = Field(default=None)

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.api_key = self.api_key or os.getenv("OPENAI_API_KEY")
        self.base_url = self.base_url or os.getenv("OPENAI_API_BASE")
        if not self.api_key or not self.base_url:
            raise ValueError("API key 和 Base URL 必须通过参数或环境变量提供")

        # 初始化 OpenAI 原生客户端
        self._client = OpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
        )

    @property
    def _llm_type(self) -> str:
        return "qwen_dashscope"

    def _convert_messages_to_openai_format(self, messages: List[BaseMessage]) -> List[Dict[str, Any]]:
        """将 LangChain 消息格式转换为 OpenAI SDK 要求的格式"""
        openai_messages = []
        for msg in messages:
            role = "user"
            if isinstance(msg, HumanMessage):
                role = "user"
            elif isinstance(msg, AIMessage):
                role = "assistant"
            elif isinstance(msg, SystemMessage):
                role = "system"
            elif isinstance(msg, ChatMessage):
                role = msg.role

            openai_messages.append({"role": role, "content": msg.content})
        return openai_messages

    def _generate(
            self,
            messages: List[BaseMessage],
            stop: Optional[List[str]] = None,
            run_manager: Optional[CallbackManagerForLLMRun] = None,
            **kwargs: Any,
    ) -> ChatResult:
        """核心生成逻辑"""

        # 1. 转换消息格式
        openai_messages = self._convert_messages_to_openai_format(messages)

        # 2. 准备请求参数 (extra_body 是关键)
        extra_body = kwargs.pop("extra_body", {})

        # 3. 调用 OpenAI SDK
        response = self._client.chat.completions.create(
            model=self.model_name,
            messages=openai_messages,
            extra_body=extra_body,  # 传入百炼特有参数
            stop=stop,
            **kwargs
        )

        # 4. 解析结果
        choice = response.choices[0]
        message = choice.message
        content = message.content

        # 5. 处理思考过程 (Reasoning Content)
        # 百炼 API 的思考内容通常在 message 的 reasoning_content 字段中（如果有）
        # 或者有时在 extra_fields 里，我们将其放入 additional_kwargs 以便后续查看
        reasoning_content = getattr(message, "reasoning_content", None)

        additional_kwargs = {}
        if reasoning_content:
            additional_kwargs["reasoning_content"] = reasoning_content

        # 6. 构造 LangChain 的返回值
        generations = [
            ChatGeneration(
                message=AIMessage(
                    content=content,
                    additional_kwargs=additional_kwargs  # 思考过程存在这里
                )
            )
        ]

        return ChatResult(generations=generations)

In [3]:
llm = QwenChatModel(model="qwen-plus")

# 思考过程

In [4]:
from IPython.display import display, Markdown

messages = [
    HumanMessage(content="请帮我解方程：x^2 - 5x + 6 = 0")
]
result = llm.invoke(messages, extra_body={'enable_thinking': True})

reasoning = result.additional_kwargs.get("reasoning_content", "")
final_answer = result.content

In [5]:
# 渲染思考过程
print("=== 思考过程 ===")
if reasoning:
    display(Markdown(reasoning))  # 加个引用符号区分
else:
    print("无思考过程")

print("\n=== 最终回答 ===")
display(Markdown(final_answer))

=== 思考过程 ===


我现在要解这个二次方程x² - 5x + 6 = 0。首先，我记得二次方程的一般解法有因式分解、求根公式、配方法等等。首先试试看能不能因式分解，因为如果能分解的话会比较简单。

首先，二次项的系数是1，所以我们可以假设这个二次式可以分解成(x + a)(x + b)的形式，其中a和b是两个数，需要满足a + b = -5（因为一次项的系数是-5），而a*b = 6（常数项是6）。不过这里要注意符号，因为原方程是x² - 5x + 6，所以应该是(x - m)(x - n) = 0，其中m和n是正数，因为常数项是正的，一次项是负的，所以两个根都是正的。

现在需要找两个数m和n，使得m + n = 5，m*n = 6。那我们可以列出6的因数对：1和6，2和3。1+6=7，不符合；2+3=5，刚好符合。所以m=2，n=3，或者m=3，n=2。

所以原方程可以分解为(x - 2)(x - 3) = 0。根据零乘积定理，当两个因子的乘积为0时，至少其中一个因子为0，所以x - 2 = 0或者x - 3 = 0，解得x=2或x=3。

为了验证是否正确，我可以把x=2代入原方程：2² - 5*2 + 6 = 4 - 10 + 6 = 0，对的。x=3代入：3² - 5*3 + 6 = 9 - 15 + 6 = 0，也对。

另外，也可以用求根公式来验证。二次方程ax² + bx + c = 0的根是x = [-b ± √(b² - 4ac)]/(2a)。这里a=1，b=-5，c=6，所以判别式D = (-5)² - 4*1*6 = 25 - 24 = 1，根号D=1，所以x = [5 ± 1]/2，即(5+1)/2=3，(5-1)/2=2，和因式分解的结果一致。

所以这个方程的解是x=2和x=3。


=== 最终回答 ===


方程 $x^2 - 5x + 6 = 0$ 的解法如下：

### 步骤 1：因式分解
观察二次项系数为 1，尝试将方程分解为 $(x - a)(x - b) = 0$ 的形式。  
需满足：
- $a + b = 5$（一次项系数的相反数）
- $a \cdot b = 6$（常数项）

符合条件的数对为 $a = 2$ 和 $b = 3$，因此方程可分解为：
$$
(x - 2)(x - 3) = 0
$$

### 步骤 2：求解根
根据零乘积定理：
$$
x - 2 = 0 \quad \text{或} \quad x - 3 = 0
$$
解得：
$$
x = 2 \quad \text{或} \quad x = 3
$$

### 验证
代入原方程验证：
- 当 $x = 2$ 时：$2^2 - 5 \cdot 2 + 6 = 4 - 10 + 6 = 0$
- 当 $x = 3$ 时：$3^2 - 5 \cdot 3 + 6 = 9 - 15 + 6 = 0$

### 结论
方程的解为：
$$
\boxed{x = 2} \quad \text{和} \quad \boxed{x = 3}
$$

# 结构化输出

In [6]:
from pydantic import BaseModel, Field


class TestSchema(BaseModel):
    reasoning: str = Field(description="简短的思考过程")
    verification_token: str = Field(description="必须填写为 'CONFIRMED_BY_SCHEMA'，不要改变它")
    answer: str = Field(description="最终的答案")
    model_name: str = Field(default="qwen-plus", description="模型名称")

In [7]:
messages = [
    HumanMessage(content="你是一个AI助手，请随便生成一个 JSON 数据"),
]
result = llm.invoke(messages, response_format={"type": "json_object"})
result.pretty_print()

================================== Ai Message ==================================

{
  "id": 12345,
  "name": "Echo Nexus",
  "version": "2.7.1",
  "is_active": true,
  "created_at": "2024-05-18T09:23:47Z",
  "tags": ["ai", "json", "demo", "generator"],
  "metadata": {
    "source": "synthetic",
    "language": "en",
    "confidence_score": 0.987
  },
  "items": [
    {
      "index": 0,
      "label": "Alpha",
      "value": 42.5,
      "enabled": true
    },
    {
      "index": 1,
      "label": "Beta",
      "value": null,
      "enabled": false
    },
    {
      "index": 2,
      "label": "Gamma",
      "value": -17,
      "enabled": true
    }
  ],
  "checksum": "a1b2c3d4e5f67890"
}


In [8]:
messages = [
    HumanMessage(content=f"你是一个AI助手，请随便生成一个 JSON 数据。"),
]
result = llm.invoke(
    messages,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "schema_name",
            "schema": TestSchema.model_json_schema()
        },
    }
)
result.pretty_print()

================================== Ai Message ==================================

{
  "answer": "这是一个随机生成的JSON数据示例。",
  "model_name": "qwen-plus",
  "reasoning": "根据要求，我生成了一个符合指定JSON Schema的结构化数据，包含所有必需字段：answer、model_name（使用默认值）、reasoning和verification_token（固定为'CONFIRMED_BY_SCHEMA'）。",
  "verification_token": "CONFIRMED_BY_SCHEMA"
}
